In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
from scipy import stats
from sklearn.preprocessing import StandardScaler
import kagglehub
import os

d:\Pablo_Data\Documentos\VSCode\deep-learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Replace these with the values found inside your downloaded kaggle.json
os.environ["KAGGLE_USERNAME"] = "pablogasco1"
os.environ["KAGGLE_API_TOKEN"] = "KGAT_bb1392f0a2f508f27c8170f1f959d0c7"

# Download the dataset using Kaggle API, but check if it already exists to avoid unnecessary downloads
dataset_path = './santander_data'
if not os.path.exists(dataset_path):
    kagglehub.competition_download('santander-customer-transaction-prediction', output_dir='./santander_data')

In [3]:
# Load dataset (Assume you've downloaded train.csv from Kaggle)
train = pd.read_csv('santander_data/train.csv')
test = pd.read_csv('santander_data/test.csv')

 # drop ID_code as it is not a feature, it is unique for each row and does not contain any useful information for the model
y_train = train.pop('target')
X_train = train.drop(columns=['ID_code'])

X_test = test.drop(columns=['ID_code'])

X_train.head()

,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,-4.9200,5.7470,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,3.1468,8.0851,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,-4.9193,5.9525,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,-5.8609,8.2450,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,6.2654,7.6784,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


In [4]:
# apply standard scaling to the features except the ID_code and target
# for autoencoders is essential to have the features on the same scale, otherwise the model will have a hard time learning the patterns in the data.
scaler = StandardScaler()

# fit scaler on the training data and transform both train and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Convert to Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.int16).unsqueeze(1)  # Add an extra dimension for targets

# Create a validation split to monitor overfitting
X_train_tensor, X_val_tensor, y_train_tensor, y_val_tensor = train_test_split(X_train_tensor, y_train_tensor, test_size=0.1, random_state=42)

batch_size = 256
train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor), batch_size=batch_size, shuffle=False)

In [5]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim=200, encoding_dim=32, noise_std=0.1):
        super().__init__()
        self.noise_std = noise_std # standard deviation of the Gaussian noise to be added during training
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, encoding_dim)
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        if self.training:
            # Add Gaussian Noise: x_noisy = x + noise
            noise = torch.randn_like(x) * self.noise_std
            x_input = x + noise
        else:
            x_input = x
            
        encoded = self.encoder(x_input)
        decoded = self.decoder(encoded)
        return decoded

In [6]:
# device configuration to use GPU if available, otherwise fallback to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DenoisingAutoencoder(input_dim=200, encoding_dim=32, noise_std=0.1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
epochs = 30

#  Early stopping parameters
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in train_loader:
        X_batch = batch[0].to(device)
        output_decoded = model(X_batch)
    
        # Loss is calculated against the CLEAN original data
        loss = criterion(output_decoded, X_batch) 

        # backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # we multiply by the batch size to get the total loss for the batch, which we will average later
        train_loss += loss.item()*X_batch.size(0)
    
    train_loss /= len(train_loader.dataset)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch[0].to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict().copy()  # Save the best model state
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

# Load the best model state
model.load_state_dict(best_model_state)

Epoch [1/30], Train Loss: 0.8727, Val Loss: 0.8455
Epoch [2/30], Train Loss: 0.8429, Val Loss: 0.8420
Epoch [3/30], Train Loss: 0.8408, Val Loss: 0.8417
Epoch [4/30], Train Loss: 0.8399, Val Loss: 0.8415
Epoch [5/30], Train Loss: 0.8391, Val Loss: 0.8415
Epoch [6/30], Train Loss: 0.8386, Val Loss: 0.8414
Epoch [7/30], Train Loss: 0.8381, Val Loss: 0.8415
Epoch [8/30], Train Loss: 0.8377, Val Loss: 0.8411
Epoch [9/30], Train Loss: 0.8374, Val Loss: 0.8414
Epoch [10/30], Train Loss: 0.8371, Val Loss: 0.8413
Epoch [11/30], Train Loss: 0.8369, Val Loss: 0.8412
Epoch [12/30], Train Loss: 0.8368, Val Loss: 0.8412
Epoch [13/30], Train Loss: 0.8365, Val Loss: 0.8409
Epoch [14/30], Train Loss: 0.8364, Val Loss: 0.8413
Epoch [15/30], Train Loss: 0.8363, Val Loss: 0.8409
Epoch [16/30], Train Loss: 0.8361, Val Loss: 0.8410
Epoch [17/30], Train Loss: 0.8361, Val Loss: 0.8411
Epoch [18/30], Train Loss: 0.8359, Val Loss: 0.8410
Epoch [19/30], Train Loss: 0.8359, Val Loss: 0.8411
Epoch [20/30], Train 

<All keys matched successfully>

In [7]:
def encode_data(model, data_loader):
    model.eval()
    features = []
    with torch.no_grad():
        for batch in data_loader:
            inputs = batch[0].to(device)
            encoded = model.encoder(inputs)
            features.append(encoded.cpu().numpy())
    return np.concatenate(features, axis=0)

X_train_encoded = encode_data(model, train_loader)
X_val_encoded = encode_data(model, val_loader)
X_test_encoded = encode_data(model, test_loader)

print(f"Original train shape: {X_train_tensor.shape} → Encoded shape: {X_train_encoded.shape}")
print(f"Original val shape: {X_val_tensor.shape} → Encoded shape: {X_val_encoded.shape}")
print(f"Original test shape: {X_test_tensor.shape} → Encoded shape: {X_test_encoded.shape}")

Original train shape: torch.Size([180000, 200]) → Encoded shape: (180000, 32)
Original val shape: torch.Size([20000, 200]) → Encoded shape: (20000, 32)
Original test shape: torch.Size([200000, 200]) → Encoded shape: (200000, 32)
